# 💻 Práctica 7: Segmentación de imágenes

En esta práctica abordaremos el problema de **segmentación de pulmones en radiografías de tórax**.

La segmentación médica es una tarea fundamental en el análisis de imágenes, ya que permite **delimitar regiones anatómicas de interés** que luego pueden ser utilizadas para diagnóstico asistido, mediciones clínicas o como paso previo a otras tareas más complejas.

En nuestro caso, vamos a implementar y entrenar una red convolucional de tipo **U-Net** utilizando el dataset **JSRT**, que contiene radiografías de tórax junto con máscaras anotadas de los pulmones.

📌 Antes de comenzar, generá una copia de este archivo en tu Drive y trabajá en esa copia.

---

## 🧭 Parte 1: Práctica guiada

En esta primera parte vamos a construir un pipeline completo de segmentación que incluye:

- Carga y preparación del dataset JSRT.
- Aumentación de datos con transformaciones controladas.
- Definición de la arquitectura de la red U-Net.
- Entrenamiento y evaluación del modelo.
- Análisis cualitativo de las segmentaciones obtenidas.

### Setup inicial

Importamos las bibliotecas que vamos a necesitar:

In [ ]:
import numpy as np
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, random_split
from collections import Counter
from sklearn.metrics import confusion_matrix
from PIL import Image

Seleccionamos el dispositivo de ejecución, prefiriendo la GPU si está disponible:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo a utilizar:", device)

### Dataset JSRT

El dataset **JSRT** (*Japanese Society of Radiological Technology*) es un conjunto de datos de imágenes de radiografías de tórax.

- 247 imágenes de radiografías de tórax en escala de grises de alta resolución.
- Máscaras de GT para distintas estructuras: pulmones, corazón, clavículas.

En esta práctica vamos a trabajar únicamente con la **segmentación de pulmones**.

Para simplificar el entrenamiento, usaremos una **versión preprocesada de 224x224 píxeles**.

#### Descarga y preparación del dataset

Para comenzar, descargá el dataset desde el siguiente enlace: [JSRT](https://drive.google.com/file/d/1zONdHfRR63YXZKXw8_w1rFSEMnuMcGCL/view?usp=sharing)

Luego, hacé lo siguiente:

1. Descomprimí el archivo .zip
2. Subí la carpeta JSRT al directorio principal de tu Google Drive.

La estructura debería quedar así:

```
Mi Unidad/
 └── JSRT/
     ├── images/
     │    ├── JPCLN001.png
     │    ├── JPCLN002.png
     │    └── ...
     └── masks/
          ├── JPCLN001.png
          ├── JPCLN002.png
          └── ...
```

#### Montar Google Drive en Colab

Ejecutá la celda que sigue para conectar con Google Drive y seguí las instrucciones:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Definimos la ruta raíz del dataset:

In [ ]:
root_dir = "/content/drive/My Drive/JSRT/"

#### Exploración inicial del dataset

Antes de avanzar, verifiquemos que los datos estén bien cargados:

In [ ]:
images_dir = os.path.join(root_dir, "images")
masks_dir = os.path.join(root_dir, "masks")

image_files = sorted(os.listdir(images_dir))
mask_files = sorted(os.listdir(masks_dir))

print(f"Cantidad de imágenes: {len(image_files)}")
print(f"Cantidad de máscaras: {len(mask_files)}")

Chequeamos que los nombres (imagen y máscara) coincidan:

In [ ]:
for i in range(5):
    print(image_files[i], mask_files[i])

Ahora visualicemos algunas imágenes junto con sus máscaras para entender mejor el problema:

In [ ]:
def visualize_sample(img_path, mask_path):
    image = Image.open(img_path).convert("L")
    mask = Image.open(mask_path).convert("L")

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(image, cmap="gray")
    ax[0].set_title("Imagen")
    ax[0].axis("off")

    ax[1].imshow(mask, cmap="gray")
    ax[1].set_title("Máscara (GT)")
    ax[1].axis("off")

    ax[2].imshow(image, cmap="gray")
    ax[2].imshow(mask, cmap="jet", alpha=0.4)
    ax[2].set_title("Overlay")
    ax[2].axis("off")

    plt.show()


In [ ]:
idx = random.randint(0, len(image_files) - 1)
visualize_sample(
    os.path.join(images_dir, image_files[idx]),
    os.path.join(masks_dir, mask_files[idx])
)

❓ Preguntas:

1. ¿Qué diferencias visuales notás entre las distintas radiografías?
2. ¿Cómo te parece que eso puede afectar la tarea de segmentación?

#### Manipulando los datos

Vamos a definir una clase específica llamada `JSRTDataset` para **cargar de forma conjunta las imágenes y sus máscaras**.

Además, esta clase nos permitirá incorporar **aumentación de datos**.

En segmentación de imágenes médicas, la aumentación debe aplicarse tanto a la imagen como a su máscara.

En particular:

- Las transformaciones **geométricas** (como rotaciones o recortes) deben aplicarse **idénticamente** tanto a la imagen como a la máscara, para no perder la correspondencia espacial.
* Las transformaciones de **intensidad** (como brillo o contraste) deben aplicarse **únicamente a la imagen**, ya que la máscara contiene etiquetas y no valores físicos.

Por otro lado, **no todas las transformaciones son válidas desde el punto de vista clínico o anatómico**, por lo que es importante seleccionar cuidadosamente qué tipo de aumentación utilizar.

En nuestro caso, vamos a considerar:

- **Rotaciones pequeñas**: para simular ligeras variaciones en la postura del paciente.
- **Flip horizontal**: para introducir variabilidad entre casos.
- **Recortes leves (crop)**: asegurando no eliminar regiones relevantes como los pulmones.
- **Ajustes de brillo y contraste**: para simular diferencias entre equipos de adquisición o condiciones de exposición.

La aumentación será opcional y podrá activarse o desactivarse según el escenario.

Es importante recordar que se aplica **solo en el conjunto de entrenamiento**. Por lo tanto, **no debe utilizarse en validación ni test**, ya que alteraría la evaluación real del modelo.


Para arrancar definimos una función que aplica las transformaciones:

In [ ]:
def apply_transforms(image, mask):
    # -- Transformaciones geométricas (imagen y máscara) --
    # Flip horizontal
    if random.random() > 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    # Rotación pequeña
    angle = random.uniform(-5, 5)
    image = TF.rotate(image, angle)
    mask = TF.rotate(mask, angle)

    # Crop leve (manteniendo casi toda la imagen)
    i, j, h, w = transforms.RandomResizedCrop.get_params(
        image, scale=(0.9, 1.0), ratio=(0.95, 1.05)
    )

    image = TF.resized_crop(image, i, j, h, w, size=(224, 224))
    mask = TF.resized_crop(mask, i, j, h, w, size=(224, 224))

    # -- Transformaciones de intensidad (solo imagen) --
    brightness_factor = random.uniform(0.8, 1.2)
    contrast_factor = random.uniform(0.8, 1.2)

    image = TF.adjust_brightness(image, brightness_factor)
    image = TF.adjust_contrast(image, contrast_factor)

    return image, mask

Ahora sí, definimos la clase `JSRTDataset`:

In [ ]:
class JSRTDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augment=False):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_files = sorted(os.listdir(images_dir))
        self.mask_files = sorted(os.listdir(masks_dir))
        self.augment = augment

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        mask_path = os.path.join(self.masks_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        # Aplicar transformaciones (si corresponde)
        if self.augment:
            image, mask = apply_transforms(image, mask)

        # Conversión a tensor
        image = TF.to_tensor(image)
        mask = TF.to_tensor(mask)

        # Normalización SOLO de la imagen
        image = TF.normalize(image, mean=[0.5], std=[0.5])

        # Binarización de la máscara
        mask = (mask > 0).float()

        return image, mask

#### División del conjunto de datos

Para entrenar y evaluar el modelo de segmentación, es fundamental que separemos los datos en entrenamiento, validación y test.

Dado que el dataset JSRT es algo pequeño (247 imágenes), vamos dividirlo de manera de mantener una cantidad razonable de ejemplos en cada conjunto.

Utilizaremos las siguientes proporciones:

- 70% entrenamiento
- 15% validación
- 15% test

Además, fijamos una semilla para que el particionado sea reproducible.

In [ ]:
torch.manual_seed(42)

# Dataset completo (sin aumentación)
full_dataset = JSRTDataset(images_dir, masks_dir, augment=False)

# Tamaños de los splits
total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

# División del dataset
train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

print(f"Total de imágenes: {total_size}")
print(f"Train: {len(train_subset)}")
print(f"Validation: {len(val_subset)}")
print(f"Test: {len(test_subset)}")

Luego, creamos los datasets, especificando sólo `augment=True` para el conjunto de entrenamiento:

In [ ]:
train_dataset = torch.utils.data.Subset(
    JSRTDataset(images_dir, masks_dir, augment=True),
    train_subset.indices
)

val_dataset = torch.utils.data.Subset(
    JSRTDataset(images_dir, masks_dir, augment=False),
    val_subset.indices
)

test_dataset = torch.utils.data.Subset(
    JSRTDataset(images_dir, masks_dir, augment=False),
    test_subset.indices
)

#### Visualizando las aumentaciones

Definimos una función auxiliar para visualizar una imagen junto con su máscara:

In [ ]:
def show_sample(image, mask, title=None):
    image = image.squeeze().cpu().numpy()
    mask = mask.squeeze().cpu().numpy()

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))

    ax[0].imshow(image, cmap="gray")
    ax[0].set_title("Imagen")
    ax[0].axis("off")

    ax[1].imshow(image, cmap="gray")
    ax[1].imshow(mask, alpha=0.4, cmap="Reds")
    ax[1].set_title("Máscara superpuesta")
    ax[1].axis("off")

    if title:
        fig.suptitle(title)

    plt.show()

Ahora mostramos algunos ejemplos del conjunto de entrenamiento (aumentación activada).

In [ ]:
idxs = random.sample(range(len(train_dataset)), k=3)

for i in idxs:
    img, msk = train_dataset[i]
    show_sample(img, msk, title=f"Ejemplo train #{i}")

Las máscaras deben cubrir correctamente los pulmones y mantenerse alineadas incluso bajo rotaciones, flips o recortes.

❓ Preguntas:

1. ¿Qué pasaría si aplicaras flip vertical en lugar de horizontal?
2. ¿Sería una transformación válida para radiografías de tórax?

#### Dataloaders

Definimos los dataLoaders para cada split con un tamaño de batch de 8 ejemplos.

In [ ]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

Recuperamos un batch y verificamos la forma del mismo:

In [ ]:
images, masks = next(iter(train_loader))

print("Imágenes:", images.shape)
print("Máscaras:", masks.shape)

❓ Preguntas:

1. ¿Por qué usamos `shuffle=True` en train pero `shuffle=False` en val y test?
2. ¿Qué trade-off implica elegir un batch size más grande o más pequeño en términos de entrenamiento y memoria?
3. ¿Qué forma esperás que tenga el tensor de máscaras? ¿Por qué tiene ese número de canales?

### U-Net

Para este problema de segmentación de imágenes médicas utilizaremos una red convolucional **U-Net**.

Esta red recibe como entrada una imagen y predice como salida su segmentación. Su arquitectura consta de un **encoder** y un **decoder**, conectados mediante **skip connections**, para producir una segmentación más precisa a nivel espacial.



La U-Net utiliza un bloque básico que consta de un par de convoluciones consecutivas con ReLU en el medio:

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

La implementación que utilizaremos en esta práctica nos permitirá personalizar:

- El número de bloques encoder–decoder (`num_blocks`)
- El número de canales iniciales (`base_channels`)
- El uso o no de skip connections (`use_skips`)

Así, podremos ver cómo diferentes decisiones de diseño impactan en la calidad de las segmentaciones resultantes.

In [ ]:
class UNet(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=32,
        num_blocks=4,
        use_skips=True
    ):
        super().__init__()

        self.num_blocks = num_blocks
        self.use_skips = use_skips

        # Encoder
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()

        channels = in_channels
        encoder_channels = []

        for i in range(num_blocks):
            out_ch = base_channels * (2 ** i)
            self.encoders.append(DoubleConv(channels, out_ch))
            self.pools.append(nn.MaxPool2d(2))
            encoder_channels.append(out_ch)
            channels = out_ch

        # Bottleneck
        self.bottleneck = DoubleConv(channels, channels * 2)
        channels = channels * 2

        # Decoder
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()

        for i in reversed(range(num_blocks)):
            out_ch = encoder_channels[i]

            self.upconvs.append(
                nn.ConvTranspose2d(channels, out_ch, kernel_size=2, stride=2)
            )

            in_ch = out_ch * 2 if use_skips else out_ch
            self.decoders.append(DoubleConv(in_ch, out_ch))

            channels = out_ch

        # Capa final
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder
        for i in range(self.num_blocks):
            x = self.encoders[i](x)
            skip_connections.append(x)
            x = self.pools[i](x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        skip_connections = skip_connections[::-1]

        for i in range(self.num_blocks):
            x = self.upconvs[i](x)

            if self.use_skips:
                skip = skip_connections[i]
                if x.shape != skip.shape:
                    x = TF.resize(x, size=skip.shape[2:])
                x = torch.cat((skip, x), dim=1)

            x = self.decoders[i](x)

        return torch.sigmoid(self.final_conv(x))

Creemos un modelo y probamos pasarle un batch de juguete para verificar que funcione bien:

In [ ]:
# Instanciamos la U-Net
model = UNet(
    in_channels=1,
    out_channels=1,
    base_channels=32,
    num_blocks=4,
    use_skips=True
).to(device)

# Batch de juguete: 2 imágenes, 1 canal (grises), 224x224
x = torch.randn(2, 1, 224, 224).to(device)

# Forward pass
out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

❓ Preguntas:

1. ¿Qué información se pierde al hacer pooling y cómo la recupera el decoder?
2. ¿Por qué el decoder duplica el número de canales al concatenar la skip connections?
3. ¿Por qué la última capa usa `sigmoid` y no `softmax`?

### Función de pérdida combinada

En este problema vamos a probar con una función de pérdida combinada para optimizar dos objetivos complementarios.

En particular, usaremos:

- **Binary Cross Entropy (BCE)**: mide el error de clasificación a nivel **local**, evaluando cada píxel de forma independiente.
- **Dice Loss**: mide el error a nivel **global**, evaluando la superposición entre la predicción y la máscara real.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (
            preds.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

class BCEDiceLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.bce = nn.BCELoss()
        self.dice = DiceLoss()

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        dice_loss = self.dice(preds, targets)

        return self.alpha * bce_loss + (1 - self.alpha) * dice_loss


El parámetro `α` controlará el peso relativo de cada término en la pérdida total:

- `α → 1`: prioriza precisión local (BCE)
- `α → 0`: prioriza forma y superposición global (Dice)

BCE sola puede producir máscaras con buen detalle local pero mala forma, mientras que Dice sola puede ignorar errores pequeños.

Al combinarlas, logramos un modelo que aprende tanto detalles locales como coherencia global en la segmentación.

❓ Preguntas:

1. ¿Cuál te parece más robusto al desbalance de clases, Dice o BCE?
2. Si los pulmones ocupan el 30% aprox. de los píxeles de la imagen, ¿qué accuracy tendría un modelo que predice siempre 0?
3. ¿Por qué para transformar el Dice en función de pérdida hacemos 1 - Dice?

### Métricas de evaluación

Para evaluar la calidad de las segmentaciones producidas por el modelo utilizaremos Dice y Accuracy (nivel píxel).

El coeficiente Dice mide la superposición entre la máscara predicha y la máscara real (ground truth, GT):

$$\text{Dice} = \frac{2|A \cap B|}{|A|+|B|}$$

Su valor va de 0 (sin superposición) a 1 (correspondencia total).

In [ ]:
def dice_score(preds, targets, smooth=1e-6):
    preds = (preds > 0.5).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    intersection = (preds * targets).sum()
    dice = (2. * intersection + smooth) / (
        preds.sum() + targets.sum() + smooth
    )
    return dice.item()

La accuracy en segmentación mide la proporción de píxeles clasificados correctamente:

$$\text{Accuracy} = \frac{\text{píxeles correctos}}{\text{total de píxeles}}$$

El la misma que utilizamos en clasificación pero calculada a nivel píxel.

In [ ]:
def pixel_accuracy(preds, targets):
    preds = (preds > 0.5).float()
    correct = (preds == targets).float().sum()
    total = torch.numel(preds)
    return (correct / total).item()

❓ Preguntas:

1. Describí un escenario donde la accuracy sea alta pero el Dice sea bajo.
3. ¿Por qué se umbraliza la predicción con `> 0.5` para calcular las métricas pero no durante el entrenamiento?

### Entrenamiento

Definimos una función para entrenar el modelo durante una época:

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

También una función para evaluación:

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()

    epoch_loss = 0.0
    dice_scores = []
    acc_scores = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            epoch_loss += loss.item()
            dice_scores.append(dice_score(outputs, masks))
            acc_scores.append(pixel_accuracy(outputs, masks))

    return (
        epoch_loss / len(loader),
        np.mean(dice_scores),
        np.mean(acc_scores)
    )

Ahora sí, entrenamos el modelo:

In [ ]:
model = UNet(
    in_channels=1,
    out_channels=1,
    base_channels=32,
    num_blocks=4,
    use_skips=True
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = BCEDiceLoss(alpha=0.5)

num_epochs = 20

for epoch in range(num_epochs):
    # Training
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Validación
    val_loss, val_dice, val_acc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

### Evaluación

Evaluamos el modelo entrenado en los datos de test:

In [ ]:
_, test_dice, test_acc = evaluate(
    model, test_loader, criterion, device
)

print("Resultados en test:")
print(f"Dice: {test_dice:.4f}")
print(f"Acc: {test_acc:.4f}")

### Análisis cualitativo: visualización de resultados

Las métricas numéricas son necesarias, pero no suficientes.

En aplicaciones médicas como segmentación es fundamental inspeccionar visualmente los resultados obtenidos.

In [ ]:
def show_prediction(model, dataset, idx):
    model.eval()

    image, mask = dataset[idx]
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(image)

    pred = (pred > 0.5).float()

    image_np = image.squeeze().cpu().numpy()
    mask_np = mask.squeeze().cpu().numpy()
    pred_np = pred.squeeze().cpu().numpy()

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    ax[0].imshow(image_np, cmap="gray")
    ax[0].set_title("Imagen")
    ax[0].axis("off")

    ax[1].imshow(image_np, cmap="gray")
    ax[1].imshow(mask_np, alpha=0.4, cmap="Greens")
    ax[1].set_title("Máscara real (GT)")
    ax[1].axis("off")

    ax[2].imshow(image_np, cmap="gray")
    ax[2].imshow(pred_np, alpha=0.4, cmap="Reds")
    ax[2].set_title("Máscara predicha")
    ax[2].axis("off")

    plt.show()

In [ ]:
for i in random.sample(range(len(test_dataset)), k=3):
    show_prediction(model, test_dataset, i)

Qué podemos analizar de las visualizaciones:

- Si los bordes pulmonares están bien definidos.
- Si hay artefactos, regiones perdidas o sobresegmentadas.
- Si hay consistencia entre pacientes.
- Si hay diferencias entre pulmones izquierdo y derecho.

❓ Preguntas:

1. ¿Qué opinás cuantitativa y cualitativamente del desempeño del modelo de segmentación?
2. ¿Preferirías un modelo que subestime o sobreestime la región pulmonar en un contexto clínico?
3. ¿Qué diferencia hay entre un error en el centro del pulmón y uno en el borde? ¿Los captura BCE? ¿Y el Dice?

---

## 🧠 Parte 2: Para resolver

En esta segunda parte vamos a a experimentar modificando alguna decisión clave dentro del pipeline de segmentación:

- Aumentación de datos
- Arquitectura de la red
- Función de pérdida

En cada caso, **compará los resultados con el modelo base** (parte guiada).

Luego, al final de los ejercicios, analizá qué modificaciones tuvieron un impacto positivo en los resultados.

### Ejercicio 1: Aumentación de datos

Volvé a entrenar el modelo comparando dos de las siguientes configuraciones:

- Sin aumentación
- Sólo aumentación geométrica (rotación + flip)
- Sólo aumentación de intensidad (brillo + contraste)

Opcionalmente, podés probar aumentaciones más agresivas (rotaciones mayores, crops más fuertes, etc.)


In [ ]:
# Escribí tu código acá (o generá copias de la colab y editá la parte guiada con las modificaciones requeridas acá)

### Ejercicio 2: Arquitectura U-Net

Partiendo del modelo base, modificá alguna de estas variables:

- Profundidad (más o menos bloques).
- Canales iniciales.
- Skip connections.


In [ ]:
# Escribí tu código acá (o generá copias de la colab y editá la parte guiada con las modificaciones requeridas acá)

### Ejercicio 3: Función de pérdida

Entrená el modelo usando dos valores distintos de $\alpha$, por ejemplo:

- Solo Dice ($\alpha = 1$)
- Solo BCE ($\alpha = 0$)
- Combinaciones intermedias: $\alpha = $ 0.3, 0.5 ó 0.7.

In [ ]:
# Escribí tu código acá (o generá copias de la colab y editá la parte guiada con las modificaciones requeridas acá)

---

## 📚 Recursos

- Python: https://docs.python.org/3/tutorial/
- NumPy: https://numpy.org/learn/
- Matplotlib: https://matplotlib.org/stable/index.html
- Google Colab: https://colab.research.google.com/

- PyTorch: https://docs.pytorch.org/tutorials/beginner/basics/intro.html
- Torchvision - datasets: https://docs.pytorch.org/vision/main/datasets.html
- Torchvision - transforms: https://docs.pytorch.org/vision/0.9/transforms.html
